<a href="https://colab.research.google.com/github/Sophieli0113/ds2002-fa26/blob/main/notebooks/07-APIs/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None
    # TODO: loop up to `tries` times

    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
    # TODO:   raise on a status that is not in RETRY_STATUSES
    # TODO:   otherwise sleep (increasing each time) and try again
    # TODO: raise if every attempt failed

    for attempt in range(tries):
            try:
                r = requests.get(BASE, params=params, timeout=15)
                if r.status_code == 200:
                    return pd.DataFrame(r.json()['daily'])
                if r.status_code not in RETRY_STATUSES:
                    raise RuntimeError(f'not retryable: {r.status_code} {r.text[:200]}')
                last = f'status {r.status_code}'
            except requests.RequestException as e:
                last = repr(e)

            if attempt < tries - 1:
                delay = 0.5 * (2 ** attempt)
                print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
                time.sleep(delay)

    raise RuntimeError(f'gave up after {tries} attempts: {last}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False

try:
    wx = get_weather('2026-09-05', '2026-10-03')    # one request for the whole range
except RuntimeError as e:
    print('weather call failed:', e)
    used_fallback = True
    wx = None

if wx is not None:
    wx['time'] = pd.to_datetime(wx['time'])
    wx = wx[wx['time'].isin(pd.to_datetime(['2026-09-05', '2026-10-03']))]

print('used fallback:', used_fallback)
wx

used fallback: False


,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
28,2026-10-03,22.0,14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
if wx is None:
    used_fallback = True
    wx = FALLBACK.copy()
    wx['time'] = pd.to_datetime(wx['time'])

# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'
wx = wx.reset_index(drop=True)

print('used fallback:', used_fallback)
wx

used fallback: False


,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
wx = (wx.rename(columns={
         'time': 'date',
         'temperature_2m_max': 'temp_max',
         'precipitation_sum': 'precip_mm',
      })
      [['date', 'temp_max', 'precip_mm', 'source']])

wx['date'] = pd.to_datetime(wx['date'])
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wet = wx.loc[wx['precip_mm'].idxmax()]
dry = wx.loc[wx['precip_mm'].idxmin()]
diff = wet['precip_mm'] - dry['precip_mm']

print(f"{wet['date'].date()} was wetter by {diff:.1f} mm of rain "
      f"({wet['precip_mm']} mm vs {dry['precip_mm']} mm on {dry['date'].date()}).")

2026-10-03 was wetter by 12.2 mm of rain (14.7 mm vs 2.5 mm on 2026-09-05).


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
from datetime import datetime

print('Endpoint:', BASE)
print('Coordinates: 38.03, -78.51')
print('Date range: 2026-09-05 to 2026-10-03')
print('Pulled:', datetime.now().strftime('%Y-%m-%d %H:%M'), 'UTC')
print('Source:', wx['source'][0])

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled: 2026-10-11 00:41 UTC
Source: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

The merge would leave all the rain values blank, so the chart would look like it never rained. No error would show up and the chart would look normal, so nobody would notice the weather download had actually failed.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
orl = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
orl = orl.rename(columns={'time': 'date', 'temperature_2m_max': 'temp_max',
                          'precipitation_sum': 'precip_mm'})
orl['date'] = pd.to_datetime(orl['date'])
orl = orl[orl['date'].isin(wx['date'])]
orl['source'] = 'open-meteo'

wx['city'] = 'Charlottesville'
orl['city'] = 'Orlando'

both = pd.concat([wx, orl], ignore_index=True)
both

,date,temp_max,precip_mm,source,city
0,2026-09-05,29.4,2.5,open-meteo,Charlottesville
1,2026-10-03,22.0,14.7,open-meteo,Charlottesville
2,2026-09-05,32.1,3.9,open-meteo,Orlando
3,2026-10-03,30.4,14.5,open-meteo,Orlando
